In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [5]:
!pip install -q mlflow dagshub

import mlflow
import dagshub

dagshub.init(repo_owner='aochi23', repo_name='ml_assn_02', mlflow=True)
mlflow.set_experiment('RandomForest_Training')

Initialized MLflow to track repo "aochi23/ml_assn_02"

Repository aochi23/ml_assn_02 initialized!

<Experiment: artifact_location='mlflow-artifacts:/365154bf6230404fa936f7c0bcafab93', creation_time=1777741445533, experiment_id='3', last_update_time=1777741445533, lifecycle_stage='active', name='RandomForest_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [6]:
model = mlflow.pyfunc.load_model("models:/RF_FullPipeline/1")

In [7]:
path = '/kaggle/input/competitions/ieee-fraud-detection/'

test_transaction = pd.read_csv(path + "test_transaction.csv")
test_identity    = pd.read_csv(path + "test_identity.csv")

In [8]:
test_df = test_transaction.merge(
    test_identity,
    on="TransactionID",
    how="left"
)

test_df.columns = test_df.columns.str.replace('-', '_')

In [10]:
sk_model = model._model_impl.sklearn_model

probs = sk_model.predict_proba(test_df)[:, 1]

In [11]:
submission = pd.DataFrame({
    "TransactionID": test_df["TransactionID"],
    "isFraud": probs
})

In [12]:
submission.head(10)

,TransactionID,isFraud
0,3663549,0.064324
1,3663550,0.222929
2,3663551,0.081765
3,3663552,0.059528
4,3663553,0.091906
5,3663554,0.098239
6,3663555,0.185140
7,3663556,0.408238
8,3663557,0.061651
9,3663558,0.234824


In [13]:
submission.to_csv("random_forest_submission.csv", index=False)